In [8]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from dotenv import load_dotenv
import os

# Carregar as variáveis do arquivo .env
load_dotenv()

host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
dbname = os.getenv("DB_NAME")
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")

try:
    # Configurar a conexão com o PostgreSQL
    url = URL.create(
        drivername="postgresql+psycopg",
        username=user,
        password=password,
        host=host,
        port=int(port) if port else 5432,
        database=dbname
    )

    engine = create_engine(url)

    # Consulta SQL
    query = """
        SELECT
            ped.id_pedido,
            prod.categoria,
            ip.quantidade,
            ip.preco_unitario
        FROM pedidos ped
        INNER JOIN itens_pedido ip
            ON ped.id_pedido = ip.id_pedido
        INNER JOIN produtos prod
            ON ip.id_produto = prod.id_produto
        WHERE ped.status = 'pago'
    """

    # Executar a consulta e criar o DataFrame
    with engine.connect() as conectar:
        df_vendas = pd.read_sql(text(query), conectar)

    print("Conexão com PostgreSQL estabelecida!")
    print("Dados carregados com sucesso!")

except Exception as e:
    print("Foi ruim, deu erro:", e)

finally:
    if "engine" in locals():
        engine.dispose()

Conexão com PostgreSQL estabelecida!
Dados carregados com sucesso!


In [9]:
display(df_vendas)

,id_pedido,categoria,quantidade,preco_unitario
0,1,Livros,4,80.92
1,1,Roupas,3,297.55
2,1,Roupas,3,61.37
3,1,Casa,4,82.37
4,3,Roupas,4,54.36
...,...,...,...,...
22298,14999,Casa,2,72.00
22299,15000,Roupas,4,150.65
22300,15000,Livros,3,93.06
22301,15000,Casa,3,78.62


In [10]:
df_vendas['faturamento'] = df_vendas['quantidade'] * df_vendas['preco_unitario']
 
display(df_vendas)

,id_pedido,categoria,quantidade,preco_unitario,faturamento
0,1,Livros,4,80.92,323.68
1,1,Roupas,3,297.55,892.65
2,1,Roupas,3,61.37,184.11
3,1,Casa,4,82.37,329.48
4,3,Roupas,4,54.36,217.44
...,...,...,...,...,...
22298,14999,Casa,2,72.00,144.00
22299,15000,Roupas,4,150.65,602.60
22300,15000,Livros,3,93.06,279.18
22301,15000,Casa,3,78.62,235.86


In [11]:
faturamento_total = df_vendas["faturamento"].sum()
total_pedidos = df_vendas["id_pedido"].nunique()
ticket_medio = faturamento_total / total_pedidos if total_pedidos else 0

def formatar_reais(valor):
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

print(f"Faturamento total: {formatar_reais(faturamento_total)}")
print(f"Pedidos pagos: {total_pedidos}")
print(f"Ticket médio: {formatar_reais(ticket_medio)}")

Faturamento total: R$ 14.632.778,50
Pedidos pagos: 8971
Ticket médio: R$ 1.631,12


In [12]:
df_categoria = df_vendas.groupby("categoria").agg(
    faturamento=("faturamento", "sum"),
    quantidade_pedidos=("id_pedido", "nunique")
).reset_index()

df_categoria["ticket_medio"] = (
    df_categoria["faturamento"]
    / df_categoria["quantidade_pedidos"]
).round(2)

display(df_categoria)

,categoria,faturamento,quantidade_pedidos,ticket_medio
0,Acessórios,1993281.45,3221,618.84
1,Casa,2497412.19,3099,805.88
2,Eletrônicos,5581419.17,2289,2438.37
3,Livros,1412002.75,4470,315.88
4,Roupas,3148662.94,4878,645.48
